In [2]:
from google.colab import files

uploaded = files.upload()

Saving placement_predict_50k Dataset.csv to placement_predict_50k Dataset.csv


In [3]:
# ============================================================
# LINEAR REGRESSION WITH FEATURE SCALING AND ENCODING
# ============================================================

# ---------- 1. IMPORT LIBRARIES ----------

import pandas as pd
import numpy as np
import time

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import r2_score


# ============================================================
# 2. UPLOAD DATASET
# ============================================================

print("Please upload your CSV file:")

uploaded = files.upload()

# Automatically get uploaded filename
file_name = list(uploaded.keys())[0]

print("\nUploaded file:", file_name)


# ============================================================
# 3. READ DATASET
# ============================================================

df = pd.read_csv(file_name)

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 4. DEFINE TARGET AND FEATURES
# ============================================================

# Target variable
y = df["CGPA"]


# Numerical columns
NUM = [
    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular",
    "PlacementStatus"
]


# Categorical columns
CAT = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs"
]


# Combine features
X = df[NUM + CAT]


# ============================================================
# 5. TRAIN / TEST SPLIT
# ============================================================

Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("\nTraining samples:", len(Xtr))
print("Testing samples :", len(Xte))


# ============================================================
# A. NUMERIC ONLY
#    CATEGORICAL COLUMNS ARE REMOVED
# ============================================================

print("\n" + "="*60)
print("A. NUMERIC ONLY")
print("="*60)

# Missing value handling
num_imputer = SimpleImputer(strategy="median")

Xtr_num = num_imputer.fit_transform(Xtr[NUM])
Xte_num = num_imputer.transform(Xte[NUM])


# Linear Regression
m1 = LinearRegression()

m1.fit(Xtr_num, ytr)

pred1 = m1.predict(Xte_num)

r2_1 = r2_score(yte, pred1)

print("Numeric only R2 =", round(r2_1, 4))


# ============================================================
# B. NUMERIC + ONE-HOT ENCODED CATEGORICAL FEATURES
# ============================================================

print("\n" + "="*60)
print("B. NUMERIC + ENCODED CATEGORICAL FEATURES")
print("="*60)


# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        drop="first",
        handle_unknown="ignore"
    ))
])


# ColumnTransformer
prep = ColumnTransformer([
    ("num", numeric_pipeline, NUM),
    ("cat", categorical_pipeline, CAT)
])


# Complete pipeline
m2 = Pipeline([
    ("prep", prep),
    ("lr", LinearRegression())
])


# Train
m2.fit(Xtr, ytr)


# Predict
pred2 = m2.predict(Xte)

r2_2 = r2_score(yte, pred2)

print("Numeric + encoded R2 =", round(r2_2, 4))


# ============================================================
# SHOW COLUMNS CREATED BY TRANSFORMER
# ============================================================

created_columns = list(
    m2.named_steps["prep"].get_feature_names_out()
)

print("\nColumns created by the transformer:")

for column in created_columns:
    print(column)

print("\nTotal transformed columns:",
      len(created_columns))


# ============================================================
# C. WHY SCALING MATTERS
#    COMPARE COEFFICIENTS
# ============================================================

print("\n" + "="*60)
print("C. COEFFICIENT COMPARISON")
print("="*60)


# Unscaled Linear Regression
raw = LinearRegression()

raw.fit(Xtr_num, ytr)


# Scaled Linear Regression
scaled_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])

scaled_model.fit(Xtr[NUM], ytr)


# ---------- Unscaled coefficients ----------

print("\nUnscaled coefficients:")
print("--------------------------------")

unscaled_coefficients = pd.Series(
    raw.coef_,
    index=NUM
)

print(
    unscaled_coefficients
    .round(4)
    .to_string()
)


# ---------- Scaled coefficients ----------

print("\nScaled coefficients:")
print("--------------------------------")

scaled_coefficients = pd.Series(
    scaled_model[-1].coef_,
    index=NUM
)

print(
    scaled_coefficients
    .round(4)
    .to_string()
)


# ============================================================
# D. WHY SCALING MATTERS
#    SGD CONVERGENCE
# ============================================================

print("\n" + "="*60)
print("D. SGD CONVERGENCE")
print("="*60)


# Clean numerical data
imputer_sgd = SimpleImputer(strategy="median")

Xtr_clean = imputer_sgd.fit_transform(
    Xtr[NUM]
)


# Scaled data
scaler_sgd = StandardScaler()

Xtr_scaled = scaler_sgd.fit_transform(
    Xtr_clean
)


# Run SGD on both datasets
for label, data in [
    ("unscaled", Xtr_clean),
    ("scaled", Xtr_scaled)
]:

    t0 = time.time()

    sgd = SGDRegressor(
        max_iter=5000,
        tol=1e-4,
        random_state=0
    )

    sgd.fit(data, ytr)

    elapsed = time.time() - t0

    print(
        f"SGD on {label:<9} "
        f"iterations={sgd.n_iter_:<5} "
        f"time={elapsed:.3f}s"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print(
    "Numeric only R2           :",
    round(r2_1, 4)
)

print(
    "Numeric + encoded R2     :",
    round(r2_2, 4)
)

print(
    "R2 improvement            :",
    round(r2_2 - r2_1, 4)
)

print(
    "Total transformed columns:",
    len(created_columns)
)

print("\nProject completed successfully!")

Please upload your CSV file:


Saving placement_predict_50k Dataset.csv to placement_predict_50k Dataset (1).csv

Uploaded file: placement_predict_50k Dataset (1).csv

Dataset loaded successfully!
Dataset shape: (50000, 32)

First 5 rows:


,StudentID,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,SGPA_Sem1,SGPA_Sem2,...,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,CGPA_Tier,PlacementStatus,IsAnomaly,Salary Package
0,1,Male,Ahmedabad,Tier2,ECE,Networking,No,No,6.02,6.54,...,0,66.7,2.2,49.4,47.8,0,Low,0,0,0.00
1,2,Female,Mumbai,Tier2,ECE,DataScience,Yes,Yes,5.84,5.12,...,0,48.2,2.4,26.7,25.8,0,Low,0,0,0.00
2,3,Male,Kolkata,Tier2,IT,DataScience,Yes,No,4.91,5.29,...,0,73.8,2.8,67.7,41.5,0,Low,1,0,3.89
3,4,Male,Jaipur,Tier1,CS,AI,No,No,7.67,8.03,...,0,69.8,2.7,66.9,48.0,0,Mid,1,0,8.37
4,5,Male,Pune,Tier2,IT,DataScience,Yes,No,8.14,8.97,...,1,73.1,2.1,71.7,61.7,1,High,1,0,18.99



Training samples: 37500
Testing samples : 12500

A. NUMERIC ONLY
Numeric only R2 = 0.9224

B. NUMERIC + ENCODED CATEGORICAL FEATURES
Numeric + encoded R2 = 0.9227

Columns created by the transformer:
num__SGPA_Sem1
num__SGPA_Sem2
num__SGPA_Sem3
num__SGPA_Sem4
num__SGPA_Sem5
num__SGPA_Sem6
num__SGPA_Sem7
num__SGPA_Sem8
num__AttendancePercent
num__Internships
num__Projects
num__Workshops
num__Certifications
num__Publications
num__AptitudeTestScore
num__SoftSkillsRating
num__CodingTestScore
num__MockInterviewScore
num__ExtraCurricular
num__PlacementStatus
cat__Gender_Male
cat__City_Bangalore
cat__City_Chennai
cat__City_Delhi
cat__City_Hyderabad
cat__City_Jaipur
cat__City_Kochi
cat__City_Kolkata
cat__City_Mumbai
cat__City_Pune
cat__CollegeTier_Tier2
cat__CollegeTier_Tier3
cat__Stream_Civil
cat__Stream_ECE
cat__Stream_EE
cat__Stream_IT
cat__Stream_Mechanical
cat__Specialisation_DataScience
cat__Specialisation_Embedded
cat__Specialisation_Networking
cat__Hostel_Yes
cat__HistoryOfBacklogs_Ye